# Zero-shot learning with CLIP (T24)

Chat DL 08, decision D20 in `docs/DECISIONS.md`. Built on the class notebook
`Interacting_with_CLIP.ipynb`: the same model (CLIP ViT-B/32 with OpenAI's
weights) and the same zero-shot rule (cosine similarity x 100 as the logits
of a softmax over one sentence per class). Nothing is trained.

1. **CIFAR-100** (the notebook's dataset): zero-shot works when the images and
   their class names are the kind of (image, text) pairs CLIP learnt from.
2. **PTB-XL**: lead II of each validation and test record drawn as a 224 x 224
   ECG-paper image, one sentence per superclass. Does CLIP's shared
   image-text space reach ECG diagnosis?

All the work is done by `src/zero_shot.py`, which writes the log
`results/logs/zeroshot.json`, the table and the figures used in the reports.
This notebook runs it and shows what it does.

Setup (once, in the project's `.venv`):

```
pip install torchvision --index-url https://download.pytorch.org/whl/cu126
pip install open_clip_torch
```

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
from IPython.display import Image, display

from src import config, zero_shot as zs
from src.data import load_cache

## 1. The model

`load_clip()` returns the image encoder, the text encoder, the image
preprocessing (resize and centre-crop to 224 x 224, normalisation) and the
tokenizer, as `clip.load("ViT-B/32")` does in the class notebook.

In [ ]:
clip_model = zs.load_clip()
n_params = sum(p.numel() for p in clip_model.model.parameters())
print(clip_model.backend, "on", clip_model.device, f"- {n_params:,} parameters")

## 2. What CLIP receives: an ECG as an image

CLIP reads images, not signals, so each record is drawn as on ECG paper:
two rows of 5 s of lead II, light grid every 0.2 s and 0.5 mV, fixed scale.
The canvas is already 224 x 224, so the preprocessing does not crop it.

In [ ]:
cache = load_cache()
renderer = zs.ECGRenderer()
rows = zs.ptbxl_rows(cache)["val"][:4]
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, r in zip(axes, rows):
    ax.imshow(renderer.render(cache["X"][r, config.LEADS.index("II")]))
    labels = [c for c, v in zip(config.CLASSES, cache["y"][r]) if v]
    ax.set_title(f"ecg_id {cache['ecg_id'][r]}: {', '.join(labels)}", fontsize=9)
    ax.axis("off")
plt.show()

## 3. Cosine similarity between ECG images and sentences

The same matrix as in the class notebook: rows = sentences, columns =
images. The five diagnostic sentences and the "modality" sentences (the
first one, "an electrocardiogram", is the right one for every image).

In [ ]:
images = renderer.images(cache["X"][rows, config.LEADS.index("II")])
texts = [zs.CLASS_PROMPTS[c] for c in config.CLASSES] + zs.MODALITY_PROMPTS
img = clip_model.image_features(images)
txt = clip_model.text_features(texts)
similarity = (txt @ img.T).numpy()

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(similarity, cmap="Blues")
ax.set_yticks(range(len(texts)), texts, fontsize=8)
ax.set_xticks(range(len(rows)), [f"ecg_id {cache['ecg_id'][r]}" for r in rows],
              fontsize=8)
for i in range(len(texts)):
    for j in range(len(rows)):
        ax.text(j, i, f"{similarity[i, j]:.2f}", ha="center", va="center",
                fontsize=8)
ax.set_title("Cosine similarity between sentences and ECG images")
plt.show()
renderer.close()

## 4. The full experiment

`zs.run()` runs both parts on the whole CIFAR-100 test set and on every
validation and test record (a few minutes on the GPU; CIFAR-100 is
downloaded once into `data/external/`), then writes the log, the table and
the figures. `zs.run(fast=True)` is a quick check (outputs end in `_fast` and
are not used in the reports). The same run from a terminal:
`python -m src.zero_shot run`.

In [ ]:
record = zs.run(clip=clip_model, cache=cache)
s = record["scalars"]
print(f"CIFAR-100 top-1, notebook prompt: {s['cifar_top_one_notebook']:.3f} "
      f"(chance {s['cifar_chance_top_one']:.3f})")
print(f"PTB-XL test images seen as 'an electrocardiogram': "
      f"{s['frac_modality_ecg_test']:.1%}")
print(f"PTB-XL test macro AUROC: {s['test_macro_auroc']:.3f} "
      f"[{s['test_macro_auroc_lo']:.3f}, {s['test_macro_auroc_hi']:.3f}]")

In [ ]:
for name in ("zeroshot_examples", "zeroshot_ptbxl"):
    display(Image(filename=str(config.FIGURES_DIR / f"{name}.png")))

## 5. Reading the result

The numbers for the reports come from `results/logs/zeroshot.json` through
`report/make_numbers.py` (alias `zeroshot`), never from this notebook.
Points to discuss (block note `docs/blocks/08_learning_strategies_comparison.md`):

- Zero-shot needs a space where inputs and class *names* meet (weekly summary
  9/10: multimodal correspondence). CLIP built it from (image, text) pairs; on
  CIFAR-100 that space contains the classes.
- Recognising *that* an image is an ECG (modality check) is different from
  reading the diagnosis from it.
- Our ECG-GPT has no text side at all: it can be transferred (T13) but cannot
  classify zero-shot. A model that could would need ECG-report pairs.